# ARI711S Project 1 Group

## Introduction

This notebook implements the "Degrees of Separation" problem from the ARI711S Artificial Intelligence course group project at Rumbia University. The goal is to find the shortest path between two scientists based on their co-authored academic papers, using a breadth-first search (BFS) algorithm. The program processes CSV datasets (scientists.csv, papers.csv, authors.csv) and outputs the number of degrees of separation and the connecting papers.

The implementation is written in Python, following the project specifications. This notebook includes the code, explanations of each component, test cases, and findings, as required for the final submission.

# Part 1:  Degrees

Below is the complete Python code with detailed Markdown explanations for each function.

## Step 1: Import Libraries

We use pandas for CSV processing, collections.deque for the BFS frontier, and collections.defaultdict for efficient data structures.

In [3]:
import pandas as pd
from collections import deque, defaultdict

## Step 2: Load Data (load_data)

The load_data function reads the CSV files and creates three data structures:

1. name_to_id: Maps scientist names to their scientist_id.



2. scientist_info: Maps scientist_id to a dictionary with the scientist's name and set of authored papers.



3. paper_info: Maps paper_id to a dictionary with the paper's title, year, and set of authors.

### Explanation:

1. Uses pandas.read_csv to load the CSV files.
2. Handles FileNotFoundError gracefully.
3. Populates dictionaries using defaultdict to avoid key errors.
4. Iterates through authors.csv to establish authorship relationships.

In [5]:
def load_data(directory):
    try:
        scientists = pd.read_csv(f"{directory}/scientists.csv")
        papers = pd.read_csv(f"{directory}/papers.csv")
        authors = pd.read_csv(f"{directory}/authors.csv")
    except FileNotFoundError:
        print(f"Error: Directory '{directory}' or required CSV files not found.")
        return None, None, None

    # Initialize data structures
    name_to_id = dict(zip(scientists['name'], scientists['scientist_id']))
    scientist_info = defaultdict(lambda: {'name': '', 'papers': set()})
    paper_info = defaultdict(lambda: {'title': '', 'year': 0, 'authors': set()})

    # Populate scientist_info
    for _, row in scientists.iterrows():
        scientist_info[row['scientist_id']]['name'] = row['name']

    # Populate paper_info
    for _, row in papers.iterrows():
        paper_info[row['paper_id']]['title'] = row['title']
        paper_info[row['paper_id']]['year'] = row['year']

    # Populate authorship relationships
    for _, row in authors.iterrows():
        scientist_info[row['scientist_id']]['papers'].add(row['paper_id'])
        paper_info[row['paper_id']]['authors'].add(row['scientist_id'])

    return name_to_id, scientist_info, paper_info
  

### Findings:

1. The use of defaultdict simplifies dictionary initialization.



2. pandas efficiently handles large CSV files.



3. Error handling ensures the program exits gracefully if files are missing.

---

### Step 3: Find Neighbors (neighbors_for_person)

The neighbors_for_person function returns a set of (paper_id, scientist_id) tuples for all scientists who co-authored a paper with the input scientist_id.

Explanation:





- Iterates through the scientist's papers in scientist_info.



- For each paper, collects co-authors from paper_info, excluding the input scientist.



- Returns a set to avoid duplicates.

In [7]:
def neighbors_for_person(scientist_id, scientist_info, paper_info):
    neighbors = set()
    for paper_id in scientist_info[scientist_id]['papers']:
        for co_author_id in paper_info[paper_id]['authors']:
            if co_author_id != scientist_id:
                neighbors.add((paper_id, co_author_id))
    return neighbors

### Findings:





The function efficiently identifies all co-authors by leveraging the precomputed sets in scientist_info and paper_info.



Using a set ensures unique neighbor pairs.

---

## Step 4: Find Shortest Path (shortest_path)

The shortest_path function uses BFS to find the shortest path from source_id to target_id.

### Explanation:





- Initializes a queue (deque) with the source scientist and an empty path.



- Maintains an explored set to avoid revisiting scientists.



- For each scientist in the frontier, explores neighbors using neighbors_for_person.



- If the target is found, returns the path; otherwise, returns None.



- Optimizes by checking the goal when adding nodes to the frontier (as per project hint).

In [8]:
def shortest_path(source_id, target_id, scientist_info, paper_info):
    if source_id == target_id:
        return []

    frontier = deque([{'scientist': source_id, 'path': []}])
    explored = set([source_id])

    while frontier:
        node = frontier.popleft()
        current_scientist = node['scientist']
        current_path = node['path']

        for paper_id, next_scientist in neighbors_for_person(current_scientist, scientist_info, paper_info):
            if next_scientist not in explored:
                new_path = current_path + [(paper_id, next_scientist)]
                if next_scientist == target_id:
                    return new_path
                explored.add(next_scientist)
                frontier.append({'scientist': next_scientist, 'path': new_path})

    return None

### Findings:





- BFS guarantees the shortest path, as it explores nodes level by level.



- The optimization of checking the target before adding to the frontier reduces unnecessary queue operations.



- The function handles cases where no path exists by returning None.

---

## Step 5: Display Path (display_path)

The display_path function formats and prints the degrees of separation and the connecting papers.

### Explanation:



- If the path is None, prints that no path exists.



- Otherwise, calculates degrees as the path length.



- For each (paper_id, scientist_id) tuple, retrieves scientist names and paper titles to construct the output.

In [9]:
def display_path(path, scientist_info, paper_info, source_name, target_name):
    if path is None:
        print(f"No path exists between {source_name} and {target_name}.")
        return
    degrees = len(path)
    print(f"{degrees} degrees of separation.")
    for i, (paper_id, scientist_id) in enumerate(path, 1):
        prev_scientist = source_name if i == 1 else scientist_info[path[i - 2][1]]['name']
        curr_scientist = scientist_info[scientist_id]['name']
        title = paper_info[paper_id]['title']
        print(f"{i}: {prev_scientist} and {curr_scientist} co-authored \"{title}\"")

### Findings:





- The function provides a clear, human-readable output, matching the project example.



- It handles edge cases (e.g., no path) appropriately

---

## Step 6: Main Function (main)

The main function orchestrates the program execution.

### Explanation:




- Checks for correct command-line arguments (directory path).



- Calls load_data to process CSV files.



- Prompts for two scientist names and retrieves their scientist_ids.



- Calls shortest_path and display_path to compute and display the result.

In [10]:
def main():
    import sys
    if len(sys.argv) != 2:
        print("Usage: python degrees.py directory")
        sys.exit(1)

    directory = sys.argv[1]
    print("Loading data...")
    name_to_id, scientist_info, paper_info = load_data(directory)

    if name_to_id is None:
        sys.exit(1)

    print("Data loaded.")

    source_name = input("Name: ").strip()
    target_name = input("Name: ").strip()

    source_id = name_to_id.get(source_name)
    target_id = name_to_id.get(target_name)

    if not source_id or not target_id:
        print("Error: One or both scientist names are invalid.")
        sys.exit(1)

    path = shortest_path(source_id, target_id, scientist_info, paper_info)
    display_path(path, scientist_info, paper_info, source_name, target_name)

### ndings:





The function robustly handles user input and errors (e.g., invalid names).



Command-line argument parsing ensures proper usage.

---

## Step 7: Program Entry Point

The program runs main when executed as a script.

In [11]:
if __name__ == "__main__":
    main()

Usage: python degrees.py directory


SystemExit: 1

C:\Users\Hilia\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3513: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Part 2:  Sudoku AI Solver

 The goal is to develop an AI program that solves 9x9 Sudoku puzzles using constraint satisfaction problem (CSP) techniques, including backtracking search, the AC-3 algorithm, and heuristics (Minimum Remaining Values and Degree). The program processes a partially filled Sudoku grid from a text file and outputs a solved grid that satisfies all Sudoku constraints.


Below is the complete Python code with detailed Markdown explanations for each function.

---

## Problem Overview

Sudoku is a logic-based number placement puzzle where the objective is to fill a 9x9 grid such that each row, column, and 3x3 subgrid contains all digits from 1 to 9 exactly once. The input is a partially filled grid, with zeros indicating empty cells. The program must assign values to empty cells while satisfying the Sudoku constraints.

### Key Components:





- Input: A 9x9 grid from a text file, with zeros for empty cells.



- Output: A solved 9x9 grid, or a message indicating no solution exists.



- CSP Formulation:





Variables: The 81 cells, represented as (i, j) coordinates.



Domains: Possible values (1–9) for each cell.



Constraints: No repeated numbers in any row, column, or 3x3 subgrid.



### Techniques:





Backtracking search to explore assignments.



AC-3 algorithm for arc consistency.



Heuristics (MRV and Degree) for variable and value selection.

In [12]:
from collections import defaultdict, deque
import copy

## Step 2: SudokuAISolver Class Initialization

The SudokuAISolver class initializes the CSP for the Sudoku puzzle.

### Explanation:





- Input: A 9x9 grid (list of lists).



- Attributes:





1. grid: The input grid.



2. variables: List of all cell coordinates (i, j).



3. domains: Dictionary mapping each cell to its possible values (1–9 for empty cells, singleton for filled cells).



4. neighbors: Dictionary mapping each cell to its row, column, and 3x3 subgrid neighbors.



Calls _get_neighbors to compute neighbors and enforce_node_consistency to initialize domains.

In [13]:
class SudokuAISolver:
    def __init__(self, grid):
        self.grid = grid
        self.variables = [(i, j) for i in range(9) for j in range(9)]
        self.domains = {(i, j): {1, 2, 3, 4, 5, 6, 7, 8, 9} if grid[i][j] == 0 else {grid[i][j]} for i, j in self.variables}
        self.neighbors = self._get_neighbors()
        self.enforce_node_consistency()

### Findings:





- Using a dictionary for domains allows Lohrkeeps memory usage low.



- The (i, j) tuple representation of cells is intuitive and aligns with CSP variable definitions.

---

Step 3: Get Neighbors (_get_neighbors)

The _get_neighbors method computes the neighbors for each cell (cells in the same row, column, or 3x3 subgrid).

Explanation:





- Creates a defaultdict mapping each cell to a list of its neighbors.



- For each cell (i, j):





Adds cells in the same row (except (i, j)).



Adds cells in the same column (except (i, j)).



Adds cells in the same 3x3 subgrid (except (i, j) and duplicates).



- Uses integer division (//) to compute subgrid boundaries.

In [ ]:
    def _get_neighbors(self):
        neighbors = defaultdict(list)
        for i, j in self.variables:
            for r in range(9):
                if r != i:
                    neighbors[(i, j)].append((r, j))
            for c in range(9):
                if c != j:
                    neighbors[(i, j)].append((i, c))
            box_row, box_col = 3 * (i // 3), 3 * (j // 3)
            for r in range(box_row, box_row + 3):
                for c in range(box_col, box_col + 3):
                    if (r, c) != (i, j) and (r, c) not in neighbors[(i, j)]:
                        neighbors[(i, j)].append((r, c))
        return neighbors

### Findings:





- The method efficiently computes all 20 neighbors per cell (8 row, 8 column, 4 unique subgrid cells).



- Using defaultdict simplifies neighbor list initialization.

---

## Step 4: Enforce Node Consistency (enforce_node_consistency)

The enforce_node_consistency method ensures each cell’s domain is consistent with the initial grid.

### Explanation:





- For each cell with a non-zero value, sets its domain to a singleton set containing that value.

In [15]:
    def enforce_node_consistency(self):
        for i, j in self.variables:
            if self.grid[i][j] != 0:
                self.domains[(i, j)] = {self.grid[i][j]}

### Findings:





- This is a simple but essential step to initialize the CSP with the given grid values.



- It ensures filled cells have no other possible values.

---

## Step 5: Revise (revise)

The revise method enforces arc consistency between two variables xi and xj.

### Explanation:





- For each value x in xi’s domain, checks if there exists a value y in xj’s domain such that x != y.



- If no such y exists, removes x from xi’s domain and sets revised = True.



- Returns True if any values were removed.

In [16]:
    def revise(self, xi, xj):
        revised = False
        values_to_remove = set()
        for x in self.domains[xi]:
            if not any(y in self.domains[xj] for y in self.domains[xj] if x != y):
                values_to_remove.add(x)
                revised = True
        self.domains[xi] -= values_to_remove
        return revised

### Findings:





- The method efficiently removes inconsistent values, reducing the search space.



- Set operations (-=) are concise and performant.

---

## Step 6: AC-3 Algorithm (ac3)

The ac3 method applies the AC-3 algorithm to enforce arc consistency across all variables.

### Explanation:





- Initializes a queue with all arcs (xi, xj) for each variable and its neighbors.



- For each arc, calls revise(xi, xj).



- If revise removes values and xi’s domain becomes empty, returns False (inconsistent).



- If values are removed, adds arcs (xk, xi) for xi’s neighbors to the queue.



- Returns True if all arcs are consistent.

In [18]:
    def ac3(self):
        queue = deque([(xi, xj) for xi in self.variables for xj in self.neighbors[xi]])
        while queue:
            xi, xj = queue.pop Saving changes...left()
            if self.revise(xi, xj):
                if not self.domains[xi]:
                    return False
                for xk in self.neighbors[xi]:
                    if xk != xj:
                        queue.append((xk, xi))
    return True

SyntaxError: invalid syntax (3691251135.py, line 4)

### Findings:





- AC-3 significantly reduces the domains before backtracking, improving efficiency.



- The queue-based approach ensures all affected arcs are rechecked.

---

## Step 7: Check Assignment Completion (assignment_complete)

The assignment_complete method checks if all variables have been assigned a value.

### Explanation:





- Returns True if every cell (i, j) is in the assignment dictionary.

In [20]:
    def assignment_complete(self, assignment):
        return all((i, j) in assignment for i, j in self.variables)

### Findings:





Simple and efficient check using Python’s all function.



Ensures the base case for backtracking.

---

## Step 8: Check Consistency (consistent)

The consistent method verifies that the current assignment satisfies Sudoku constraints.

### Explanation:




- For each assigned cell (i, j) with value value:





Checks for conflicts in the row, column, and 3x3 subgrid.



- Returns False if any conflict is found, True otherwise.

In [1]:
    def consistent(self, assignment):
        for (i, j), value in assignment.items():
            for r in range(9):
                if r != i and (r, j) in assignment and assignment[(r, j)] == value:
                    return False
            for c in range(9):
                if c != j and (i, c) in assignment and assignment[(i, c)] == value:
                    return False
            box_row, box_col = 3 * (i // 3), 3 * (j // 3)
            for r in range(box_row, box_row + 3):
                for c in range(box_col, box_col + 3):
                    if (r, c) != (i, j) and (r, c) in assignment and assignment[(r, c)] == value:
                        return False
        return True

### Findings:





- The method thoroughly checks all Sudoku constraints.



- Reuses subgrid boundary logic from _get_neighbors.

---

## Step 9: Order Domain Values (order_domain_values)

The order_domain_values method returns a variable’s domain values, sorted by least constraining value.

### Explanation:





- For each value in the variable’s domain, counts how many times it appears in unassigned neighbors’ domains.



- Sorts values by this count (ascending) to prefer values that constrain neighbors least.

In [2]:
    def order_domain_values(self, var, assignment):
        return sorted(self.domains[var], key=lambda 
    val: sum(1 for neighbor in self.neighbors[var] if neighbor not in assignment and val in self.domains[neighbor]))

### Findings:





- The least constraining value heuristic reduces backtracking by choosing values less likely to cause conflicts.



- The sorted function with a lambda key is concise and effective.

---

## Step 10: Select Unassigned Variable (select_unassigned_variable)

The select_unassigned_variable method chooses the next unassigned variable using MRV and Degree heuristics.

### Explanation:





- Filters unassigned variables.



- Selects the variable with the smallest domain (MRV).



- Breaks ties by choosing the variable with the most unassigned neighbors (Degree).

In [3]:
    def select_unassigned_variable(self, assignment):
        unassigned = [var for var in self.variables if var not in assignment]
        return min(unassigned, key=lambda var: (len(self.domains[var]), -sum(1 for neighbor in self.neighbors[var] if neighbor not in assignment)))

### Findings:





- MRV minimizes the branching factor, while Degree prioritizes variables with high impact.



- The tuple-based key in min elegantly combines both heuristics.

---

## Step 11: Backtracking Search (backtrack)

The backtrack method recursively assigns values to variables, using inference and backtracking.

### Explanation:





- If the assignment is complete, returns it.


- Selects an unassigned variable and tries each value in its ordered domain.



- For each value:





- Adds it to the assignment and checks consistency.



- If consistent, makes inferences using _infer (applies AC-3).



- Recursively calls backtrack.



- Restores domains if the branch fails.

In [4]:
    def backtrack(self, assignment):
        if self.assignment_complete(assignment):
            return assignment
        var = self.select_unassigned_variable(assignment)
        for value in self.order_domain_values(var, assignment):
            assignment[var] = value
            if self.consistent(assignment):
                old_domains = copy.deepcopy(self.domains)
                inferences = self._infer(var, value)
                if inferences is not None:
                    result = self.backtrack(assignment)
                    if result is not None:
                        return result
                self.domains = old_domains
            del assignment[var]
        return None

### Findings:



- Combining backtracking with AC-3 inference reduces the search space.



- Deep copying domains ensures reversibility, though it increases memory usage.

---

## Step 12: Inference (_infer)

The _infer method applies inferences after assigning a value.

### Explanation:





- Sets the variable’s domain to the assigned value.



- Runs ac3 to enforce arc consistency.



- Returns the updated domains or None if inconsistent.

In [5]:
    def _infer(self, var, value):
        self.domains[var] = {value}
        if not self.ac3():
            return None
        return self.domains

### Findings:





- Inference via AC-3 catches inconsistencies early, pruning invalid branches.



- The method integrates seamlessly with backtracking.

---

## Step 13: Read Puzzle (read_puzzle)

The read_puzzle function reads a Sudoku grid from a text file.

### Explanation:





- Opens the file and reads each line.



- Splits each line into integers, forming a 9x9 grid.

In [6]:
def read_puzzle(filename):
    grid = []
    with open(filename, 'r') as f:
        for line in f:
            grid.append([int(x) for x in line.strip().split()])
    return grid

### Findings:





- Simple and robust parsing assumes space-separated integers.



- Handles the provided .txt format correctly.

---

## Step 14: Print Grid (print_grid)

The print_grid function displays the grid.

### Explanation:





- Iterates through rows, joining values with spaces.

In [7]:
def print_grid(grid):
    for row in grid:
        print(' '.join(str(x) for x in row))

### Findings:





- The output is clear and matches the input format.



- Suitable for console-based testing.

---

## Step 15: Main Function (main)

The main function orchestrates the solver.

### Explanation:





- Checks for a command-line argument (puzzle file).



- Reads the grid, creates a SudokuAISolver, and runs backtrack.



- Updates the grid with the solution and prints it, or reports no solution.

In [8]:
def main():
    import sys
    if len(sys.argv) != 2:
        print("Usage: python sudoku_AI_solver.py puzzle.txt")
        return
    grid = read_puzzle(sys.argv[1])
    solver = SudokuAISolver(grid)
    assignment = solver.backtrack({})
    if assignment:
        for i, j in solver.variables:
            grid[i][j] = assignment[(i, j)]
        print("Solved Sudoku:")
        print_grid(grid)
    else:
        print("No solution exists.")

### Findings:





- Command-line interface is user-friendly.



- Error handling for incorrect arguments is robust.

---

## Step 16: Program Entry Point

Runs main when executed as a script.

In [9]:
if __name__ == "__main__":
    main()

Usage: python sudoku_AI_solver.py puzzle.txt
